<a href="https://colab.research.google.com/github/marcinsawinski/scikit-learn-mooc/blob/main/workbooks/m7_01_baselines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📒 Workbook M7.01 — Is this model better than nothing?

In the lab we compared models with **baselines** that ignore the
features, and looked at the whole distribution of cross-validation
scores before declaring a winner. In this workbook you do the same for
a model that follows **Parkinson's disease from voice recordings**. You
will ask what the R² of a baseline must be, compare two models fold by
fold, and scramble the target to see what "no information" looks like.
The model will pass every test, and the last section explains why you
should not believe it yet.

**How this workbook works**

- The code is written for you. You do not need to write code, but you need
  to **understand every step**.
- Every `____` is a gap. Fill it in with the value asked for in the text
  **above** the cell. The error `NameError: name '____' is not defined`
  means: a gap is still empty.
- `check(...)` tells you whether your value is right (✅) or stops with a
  hint (❌).
- `commit(...)` records a **prediction** before you see the result. Be
  honest: a wrong prediction costs nothing, it is how you learn.
- Questions marked **🧠 Think** are not checked here. You will need the
  answers for the Moodle quiz and for the discussion at the start of the
  next lab.

In [ ]:
#@title Setup — run this cell first (no need to read it)
import hashlib
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

ANSWERS = {
    "folds_won_by_ridge": "e7f6c011776e8db7",
    "gap_against_spread": "44934275b4bab1b3",
    "r2_of_mean_prediction": "5feceb66ffc86f38",
    "scrambled_model_lands": "a638fc24f1b58a54",
    "what_was_measured": "4530af5bffbfdad7"
}

def _normalize(value):
    if isinstance(value, float) and value.is_integer():
        value = int(value)
    if isinstance(value, float):
        value = round(value, 2)
    if isinstance(value, (list, tuple, set)):
        return ",".join(sorted(str(v).strip().lower() for v in value))
    return str(value).strip().lower()


def _hash(value):
    return hashlib.sha256(_normalize(value).encode()).hexdigest()[:16]


def check(name, value, hint="Re-read the text above this cell and try again."):
    """Stop with a hint unless `value` is the expected answer for `name`."""
    if _hash(value) != ANSWERS[name]:
        raise ValueError(f"❌ {name} = {value!r} is not right. {hint}")
    print(f"✅ {name}")


def check_score(name, value, actual):
    """Stop with a hint unless `value` is `actual` rounded to 2 decimals.

    For results that differ slightly between computers (e.g. k-NN ties).
    """
    if not isinstance(value, (int, float)) or abs(value - round(actual, 2)) > 0.005:
        raise ValueError(
            f"❌ {name} = {value!r} is not right. "
            "Read the score printed above and round it to 2 decimals."
        )
    print(f"✅ {name}")


def commit(prediction, options):
    """Record a prediction before seeing the result."""
    if prediction not in options:
        raise ValueError(f"Choose one of {options} first.")
    print(f"📌 Your prediction: {prediction!r}. Now run the next cell and compare.")

## 1. Data and target

42 people in the early stage of Parkinson's disease took part in a
six-month trial. At home, about once a week, each of them recorded their
voice several times with a medical device. Each row of the data is **one
recording**.

- `subject`: the number of the patient (1 to 42);
- `age`, `sex` (0 = man, 1 = woman);
- `test_time`: days since the patient joined the trial;
- 16 measurements of the voice: variations of pitch (`jitter_…`), of
  loudness (`shimmer_…`), noise and other signal measures;
- `total_updrs`: the severity of the symptoms on the clinical scale used
  by neurologists (higher is worse). Doctors assessed each patient three
  times during the trial; the value for each recording day was filled in
  between those visits.

The hope: follow the disease **from home, with a microphone**, without
a visit to the hospital. We predict `total_updrs` from the voice
measurements, the age and the sex.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

recordings = pd.read_csv(
    "https://raw.githubusercontent.com/marcinsawinski/scikit-learn-mooc/refs/heads/main/datasets/parkinsons_telemonitoring.csv"
)
features = [
    name for name in recordings.columns if name not in ("subject", "test_time", "motor_updrs", "total_updrs")
]
data = recordings[features]
target = recordings["total_updrs"]
print(f"{len(recordings)} recordings, {recordings['subject'].nunique()} patients, {len(features)} features")
print(f"total_updrs from {target.min():.0f} to {target.max():.0f}, mean {target.mean():.1f}")
recordings[["subject", "age", "sex", "test_time", "jitter_percent", "shimmer", "total_updrs"]].head()

We evaluate everything with the same 6-fold cross-validation, shuffled,
and two scores: the mean absolute error in UPDRS points (lower is
better) and R² (higher is better).

In [ ]:
from sklearn.model_selection import KFold, cross_validate

cv = KFold(n_splits=6, shuffle=True, random_state=0)


def evaluate(model, target=target):
    scores = cross_validate(model, data, target, cv=cv, scoring=("neg_mean_absolute_error", "r2"))
    return pd.DataFrame({"error": -scores["test_neg_mean_absolute_error"], "r2": scores["test_r2"]})

## 2. Baselines

A `DummyRegressor` never looks at the features: it predicts the same
value for every recording, by default the mean of the training target.

✏️ R² compares a model with the strategy "always predict the mean of
the target": 1 means perfect predictions, and a model that does no
better than predicting the mean gets… which value?

In [ ]:
r2_of_mean_prediction = ____
check("r2_of_mean_prediction", r2_of_mean_prediction)

In [ ]:
from sklearn.dummy import DummyRegressor

results = {
    "dummy (mean)": evaluate(DummyRegressor(strategy="mean")),
    "dummy (median)": evaluate(DummyRegressor(strategy="median")),
}
pd.DataFrame({name: scores.mean() for name, scores in results.items()}).T.round(2)

The two baselines are close here, because the target is not lopsided
(compare with the football wages of workbook M6.01, where the median was
much better). An error of about **8.6 UPDRS points** is what we get for
free. A negative R² is possible: it means "worse than predicting the
mean".

## 3. A model against the baseline

Our model is a histogram gradient boosting (module 6).

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

model = HistGradientBoostingRegressor(random_state=0)
results["gradient boosting"] = evaluate(model)

fig, ax = plt.subplots(figsize=(7, 3))
for name in ["dummy (mean)", "gradient boosting"]:
    ax.plot(results[name]["error"], [name] * 6, "o", alpha=0.7)
ax.set(xlabel="error on each of the 6 folds (UPDRS points)", xlim=(0, 10))
ax.margins(y=0.6)
pd.DataFrame({name: scores.mean() for name, scores in results.items()}).T.round(2)

✏️ Compare the gap between the model and the baseline with the spread of
the points of one model across folds. Is the gap `"much larger"` than
the spread, `"about as large"` or `"smaller"`?

In [ ]:
gap_against_spread = ____
check("gap_against_spread", gap_against_spread)

An error of 2.4 points against 8.7, an R² above 0.9, and almost no
variation between folds. Every recording day, the model seems to know
the state of the patient almost as well as a neurologist.

### ➕ Going further: a close race, judged fold by fold

Not every comparison is this clear. A linear model (`Ridge`) is only a
little better than the baseline.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

results["ridge"] = evaluate(make_pipeline(StandardScaler(), Ridge()))
close_race = pd.DataFrame(
    {"dummy (mean)": results["dummy (mean)"]["error"], "ridge": results["ridge"]["error"]}
).rename_axis("fold")
close_race["difference"] = close_race["dummy (mean)"] - close_race["ridge"]
print(f"dummy: {close_race['dummy (mean)'].mean():.2f} ± {close_race['dummy (mean)'].std():.2f}")
print(f"ridge: {close_race['ridge'].mean():.2f} ± {close_race['ridge'].std():.2f}")
close_race.round(2)

Reading only "8.66 ± 0.26 against 8.05 ± 0.20", the two ranges almost
touch. But both models were tested on **the same six folds**, and some
folds are simply harder than others for everybody. The fair comparison
is **fold by fold**: the `difference` column.

✏️ In how many of the 6 folds is the ridge better than the dummy?

In [ ]:
folds_won_by_ridge = ____
check("folds_won_by_ridge", folds_won_by_ridge, hint="Count the folds where the difference is positive.")

A small advantage that shows up in every fold is more convincing than
the overlap of two `±` suggests. When comparing two models, always
evaluate them on the same splits and look at the differences.

## 4. ➕ Going further: what "no information" looks like

The baseline ignores the features. Here is another way to remove the
information: keep the features and **scramble the target**, so that each
recording gets the UPDRS score of a random other recording. The model
trains as usual, but there is nothing left to learn.

🔮 **Predict first.** Trained and tested on scrambled targets, the
gradient boosting will have an error **"still close to 2.4"**, **"about
that of the dummy"** or **"far worse than the dummy"**?

In [ ]:
my_prediction = ____
commit(my_prediction, ["still close to 2.4", "about that of the dummy", "far worse than the dummy"])

In [ ]:
rng = np.random.default_rng(0)
scrambled_errors = []
for _ in range(5):
    scrambled_target = pd.Series(rng.permutation(target.to_numpy()), index=target.index)
    scrambled_errors.append(evaluate(model, target=scrambled_target)["error"].mean())
print("error of the model on 5 scrambled targets:", np.round(scrambled_errors, 2))
print(f"error of the model on the real target:      {results['gradient boosting']['error'].mean():.2f}")
print(f"error of the dummy on the real target:      {results['dummy (mean)']['error'].mean():.2f}")

✏️ Where does the model land when the link between features and target
is destroyed? Write `"near the real model"` or `"near the dummy"`.

In [ ]:
scrambled_model_lands = ____
check("scrambled_model_lands", scrambled_model_lands)

None of the five scrambled versions comes anywhere near 2.4. This is a
**permutation test**: if the real score were a fluke, some scrambled
targets would do as well. scikit-learn automates it with
`permutation_test_score`, which repeats the scrambling many times and
reports the share of scrambled scores that beat the real one (a
*p-value*).

So: far better than both baselines, stable across folds, and impossible
to reproduce by chance. By every test of this workbook, the model is
excellent.

## 5. One more question

All these tests compare the model with something worse **on the same
test folds**. None of them asks whether the test folds themselves are
fair. Let us look at who is in the first one.

In [ ]:
train_rows, test_rows = next(cv.split(data))
patients_train = set(recordings["subject"].iloc[train_rows])
patients_test = set(recordings["subject"].iloc[test_rows])
print(f"patients with recordings in the training part: {len(patients_train)}")
print(f"patients with recordings in the test part:     {len(patients_test)}")
print(f"patients in both:                              {len(patients_train & patients_test)}")

✏️ What has this cross-validation measured? The ability to predict the
score for `"new recordings of patients the model already knows"` or for
`"patients the model has never seen"`?

In [ ]:
what_was_measured = ____
check("what_was_measured", what_was_measured)

Every patient is on both sides of every split. You met this situation
with the concrete recipes of workbook M2.01 and the copied wines of
workbook M5.03.

A baseline, a fold-by-fold comparison and a permutation test all answer
the question "is the model better than nothing **on this test**?". They
cannot tell you whether the test resembles the way the model will be
used. That is the subject of the next workbook, and the result there is
not the one you expect.

🧠 **Think.**

1. A hospital wants to use this model for patients who join next year.
   Is the error of 2.4 points the one they should expect?
2. The model receives `age` and `sex`. In a group of 42 people, how many
   share both the same age and the same sex? What could the model do
   with that?

## 🎤 Before the next lab

1. Do the **Moodle quiz** for this workbook (mandatory, not graded).
2. At the start of the next lab, some of you will be asked to present
   (3 minutes, using your notebook on the projector) **one** of:
   - what a dummy regressor predicts, and what its R² is, and why;
   - the close race between the ridge and the dummy: why the comparison
     is made fold by fold;
   - the scrambled targets: what the experiment shows and what a
     permutation test is;
   - why the excellent score of the model cannot be trusted yet.